# Bash Control Flow

## Quick notes

### What this covers
Making decisions and repeating work: `test`, `[ ]` and `[[ ]]`, file / string / numeric tests, `if` / `elif` / `else`, `case`, `for` (lists, C-style, globs, `"$@"`), `while` / `until`, reading lines safely, `break` / `continue N`, `select`, `(( ))` loops, and control flow driven directly by exit statuses.

### Key points
- `if` does not test a condition, it runs a command and looks at its exit status. Status `0` means true, anything else means false.
- `[ ]` is the command `test` (spaces around the brackets are required). `[[ ]]` is a bash keyword with safer parsing and extra features (patterns, regex, `&&`, `||`).
- Numeric tests use `-eq -ne -lt -le -gt -ge`. String tests use `= != < >` (and `-z`, `-n`). Mixing them up is a classic bug.
- `case` matches a word against glob-style patterns. `;;` ends a branch. `a|b)` means "a or b".
- `for x in "$@"` is the correct way to loop over arguments. `for f in *` is the correct way to loop over files.
- `while IFS= read -r line` is the correct way to loop over lines. Never loop over `$(cat file)` or `$(ls)`.
- `break N` and `continue N` can leave or skip several nested loops.

### Gotchas (the part worth revising twice)
- `[ $x = y ]` breaks when `x` is empty or has spaces. Quote it: `[ "$x" = y ]`.
- `[ -n $x ]` is true even when `x` is empty (unquoted, it becomes `[ -n ]`).
- `[[ ]]` does not exist in strict POSIX `sh` (for example `dash` on Debian/Ubuntu). On macOS `/bin/sh` is bash, so it appears to work there.
- `cmd | while read ...` runs the loop in a subshell: variables set inside are lost afterwards.
- `for f in *.txt` with no match gives you the literal text `*.txt`.
- `a && b || c` is not if/else (see `01-Fundamentals`).
- `(( i++ ))` returns status 1 when `i` is 0, which stops a `set -e` script on newer bash (not on 3.2, see section 17).
- A last line without a trailing newline is skipped by plain `while read` unless you handle it.

### When it matters
Any script that branches on input, validates arguments, retries something or processes a list of files. Most real-world bugs live in quoting inside conditions and in loops over filenames or command output.

### One-line recap example
```bash
for f in ./*.txt; do [ -e "$f" ] || continue; echo "$f"; done
```

Related notebooks: `01-Fundamentals` (exit statuses, quoting), `03-Functions-And-Arguments` (`"$@"`, `shift`), `04-Strings-And-Arrays` (arrays to loop over), `03-Files-And-System/01-File-System-Ops` (files and `find`).

## 01. `if` runs a command and looks at its status
- The condition after `if` is any command. Status `0` takes the `then` branch, non-zero takes `elif` / `else`.
- That is why `if grep -q pattern file; then` works with no brackets at all.
- The `then` body cannot be empty. Use `:` (the no-op command) as a placeholder.
- `if`, `then`, `else`, `fi` are keywords, so `then` needs a newline or `;` before it.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
if true; then echo "true -> then branch"; fi
if false; then echo "never"; else echo "false -> else branch"; fi
if grep -q "bash" <<< "I like bash"; then echo "grep found it (status 0)"; fi
if grep -q "perl" <<< "I like bash"; then echo "never"; else echo "grep did not find it (status 1)"; fi
if true; then :; fi          # empty body needs the no-op
echo "done"

true -> then branch
false -> else branch
grep found it (status 0)
grep did not find it (status 1)
done


## 02. `test`, `[ ]` and `[[ ]]`: what they actually are
- `[` is a command, a synonym of `test` where the last argument must be `]`. That is why the spaces matter.
- Because it is a normal command, the shell has already expanded and split its arguments. That is where the quoting bugs come from.
- `[[ ]]` is part of the shell grammar. The shell parses the expression itself, so it does not split or glob variables inside it.
- `[` exists in bash as a builtin and also as a file (`/bin/[`). Both follow the same rules, so `[ ]` works the same in any POSIX shell.

In [3]:
%%bash
echo "[  is a: $(type -t [)"
echo "test is a: $(type -t test)"
echo "[[ is a: $(type -t '[[')"
[ -x '/bin/[' ] && echo "an external /bin/[ file exists too"
test 5 -gt 3; echo "test 5 -gt 3 -> $?"
[ 5 -gt 3 ]; echo "[ 5 -gt 3 ] -> $?"
[[ 5 -gt 3 ]]; echo "[[ 5 -gt 3 ]] -> $?"

[  is a: builtin
test is a: builtin
[[ is a: keyword
an external /bin/[ file exists too
test 5 -gt 3 -> 0
[ 5 -gt 3 ] -> 0
[[ 5 -gt 3 ]] -> 0


In [4]:
%%bash expect_fail
[5 -gt 3]

script: line 1: [5: command not found
[exit status 127]


## 03. `[ ]` gotchas: empty values, spaces, `-a`/`-o`, `<`
- Unquoted, an empty variable disappears and the test sees too few arguments.
- A value with spaces becomes several arguments: `[ $name = "Ada Lovelace" ]` is a syntax error.
- `[ -n $x ]` with `x` empty is `[ -n ]`, which is a one-argument test: "is the string `-n` non-empty?" and is true.
- Inside `[ ]`, `<` and `>` are redirections unless escaped. Prefer `[[ ]]` for string ordering.
- `-a` and `-o` inside `[ ]` are ambiguous and deprecated. Use `[ A ] && [ B ]` or `[ A ] || [ B ]`.
- `==` inside `[ ]` is a bash extension. POSIX `[ ]` only has `=`.

In [5]:
%%bash expect_fail
empty=""
[ $empty = foo ]

script: line 2: [: =: unary operator expected
[exit status 2]


In [6]:
%%bash expect_fail
name="Ada Lovelace"
[ $name = "Ada Lovelace" ]

script: line 2: [: too many arguments
[exit status 2]


In [7]:
%%bash
empty=""
[ "$empty" = foo ] && echo "never"; echo "quoted: safe, status $?"
[ -n $empty ] && echo "[ -n \$empty ] unquoted is TRUE even though empty (bug)"
[ -n "$empty" ] || echo "[ -n \"\$empty\" ] quoted is false (correct)"
[ -z "$empty" ] && echo "-z is the clear way to ask: is it empty?"

quoted: safe, status 1
[ -n $empty ] unquoted is TRUE even though empty (bug)
[ -n "$empty" ] quoted is false (correct)
-z is the clear way to ask: is it empty?


In [8]:
%%bash
# string order in [ ] needs escaping; [[ ]] is simpler
[ apple \< banana ] && echo "[ apple \< banana ] true"
[[ apple < banana ]] && echo "[[ apple < banana ]] true"
# combine tests with && / || between brackets
a=3
[ "$a" -gt 1 ] && [ "$a" -lt 5 ] && echo "1 < a < 5"

[ apple \< banana ] true
[[ apple < banana ]] true
1 < a < 5


## 04. `[[ ]]`: what you gain
- No word splitting or globbing of unquoted variables inside. `[[ $x = foo ]]` is safe even if `x` is empty or has spaces.
- `&&`, `||`, `!` and parentheses work inside.
- `==` and `!=` do pattern matching when the right side is unquoted: `[[ $f == *.txt ]]`. Quote the right side for a literal comparison.
- `=~` does regular-expression (ERE) matching and fills `BASH_REMATCH`.
- The cost: it is bash / ksh / zsh only (see the next section).

In [9]:
%%bash
name="Ada Lovelace"
[[ $name = "Ada Lovelace" ]] && echo "unquoted left side is safe"
unset nothing
[[ -z $nothing ]] && echo "unset variable, no quoting needed"
f="report.txt"
[[ $f == *.txt ]] && echo "pattern match: $f ends in .txt"
[[ $f == "*.txt" ]] || echo "quoted right side is literal: not equal to the text *.txt"
[[ $f == r* && $f != *.md ]] && echo "&& inside [[ ]]"

unquoted left side is safe
unset variable, no quoting needed
pattern match: report.txt ends in .txt
quoted right side is literal: not equal to the text *.txt
&& inside [[ ]]


In [10]:
%%bash
# regular expressions: keep the pattern in a variable (portable, avoids quoting problems)
re='^([a-z]+)-([0-9]+)$'
id="order-4521"
if [[ $id =~ $re ]]; then
  echo "whole: ${BASH_REMATCH[0]}  word: ${BASH_REMATCH[1]}  number: ${BASH_REMATCH[2]}"
fi
[[ "abc" =~ ^[0-9]+$ ]] || echo "no match -> status 1"

whole: order-4521  word: order  number: 4521
no match -> status 1


## 05. When `[[ ]]` is not available (strict `sh`)
- On Debian/Ubuntu, `/bin/sh` is `dash`, which has no `[[ ]]`. A script with `#!/bin/sh` and `[[ ]]` fails there.
- On macOS, `/bin/sh` is bash in POSIX mode, where `[[ ]]` still works. A script can pass on a Mac and fail in CI or on a server.
- Rule: if the shebang says `bash`, use `[[ ]]` freely. If it says `sh`, stick to `[ ]`, `case` and POSIX features.
- Portable replacements: pattern match with `case`, regex with `grep -E` or `expr`.

In [11]:
%%bash
echo "sh on this machine reports bash version: $(sh -c 'echo ${BASH_VERSION%%.*}')"
if [ -x /bin/dash ]; then
  /bin/dash -c '[[ a == a ]] && echo "works"' 2>&1 | sed 's/^[^:]*: //'
else
  echo "(dash not installed here; skipped)"
fi

sh on this machine reports bash version: 3
1: [[: not found


In [12]:
%%bash
# the portable way: case does pattern matching in any POSIX shell
f="report.txt"
case "$f" in
  *.txt) echo "case: $f is a .txt file" ;;
esac
# a portable regex-ish test
printf '%s\n' "$f" | grep -Eq '^[a-z]+\.txt$' && echo "grep -E: matches"

case: report.txt is a .txt file
grep -E: matches


## 06. File tests
- `-e` exists, `-f` regular file, `-d` directory, `-L` symlink, `-s` non-empty file, `-r -w -x` permissions for you.
- `a -nt b` newer than, `a -ot b` older than, `a -ef b` same file.
- `-e` is true for a directory too. `-f` is the strict "regular file" test. A broken symlink fails `-e` but passes `-L`.
- Always quote the filename.

In [13]:
%%bash
mkdir tdir
echo content > file.txt
: > empty.txt
ln -s file.txt link.txt
ln -s missing.txt broken.txt
chmod 644 file.txt
for p in file.txt tdir empty.txt link.txt broken.txt nothing.txt; do
  r=""
  [ -e "$p" ] && r="$r e"
  [ -f "$p" ] && r="$r f"
  [ -d "$p" ] && r="$r d"
  [ -L "$p" ] && r="$r L"
  [ -s "$p" ] && r="$r s"
  printf '%-12s %s\n' "$p" "${r:-(none)}"
done

file.txt      e f s
tdir          e d s
empty.txt     e f
link.txt      e f L s
broken.txt    L
nothing.txt  (none)


In [14]:
%%bash
chmod +x file.txt
[ -x file.txt ] && echo "file.txt is executable now"
chmod -x file.txt
[ -x file.txt ] || echo "and not any more"
# timestamps with fixed values (touch -t is portable: [[CC]YY]MMDDhhmm)
touch -t 202001010000 old.txt
touch -t 202101010000 new.txt
[ new.txt -nt old.txt ] && echo "new.txt -nt old.txt"
[ old.txt -ot new.txt ] && echo "old.txt -ot new.txt"
[ link.txt -ef file.txt ] && echo "link.txt -ef file.txt (same underlying file)"

file.txt is executable now
and not any more
new.txt -nt old.txt
old.txt -ot new.txt
link.txt -ef file.txt (same underlying file)


## 07. String tests
- `-z STR` true if empty, `-n STR` true if not empty.
- `=` and `==` equal, `!=` different. In `[[ ]]` the right side is a pattern unless quoted.
- `<` and `>` compare in the current locale's order (the notebook runs with `LC_ALL=C`, so uppercase sorts before lowercase).
- Comparing numbers as strings gives surprises: `"10" < "9"` is true.

In [15]:
%%bash
s=""
[ -z "$s" ] && echo "-z: empty"
s="x"
[ -n "$s" ] && echo "-n: not empty"
[ "abc" = "abc" ] && echo "= equal"
[ "abc" != "abd" ] && echo "!= different"
[[ "Zebra" < "apple" ]] && echo "in the C locale, uppercase Z sorts before lowercase a"
[[ "10" < "9" ]] && echo "as strings, 10 < 9 is TRUE (compare numbers with -lt)"
[ 10 -lt 9 ] || echo "as numbers, 10 -lt 9 is false"

-z: empty
-n: not empty
= equal
!= different
in the C locale, uppercase Z sorts before lowercase a
as strings, 10 < 9 is TRUE (compare numbers with -lt)
as numbers, 10 -lt 9 is false


## 08. Numeric tests
- `-eq -ne -lt -le -gt -ge` compare integers.
- `5 -eq 05` is true (numbers) but `[[ 5 == 05 ]]` is false (different strings).
- Non-integers (`3.0`, `abc`, empty) give an error and status 2, not "false".
- For arithmetic comparisons, `(( a > b ))` is easier to read and allows expressions (see section 17).

In [16]:
%%bash
[ 5 -eq 05 ] && echo "5 -eq 05: true (numeric)"
[[ 5 == 05 ]] || echo "5 == 05: false (string)"
a=7
[ "$a" -ge 5 ] && echo "7 -ge 5"
(( a > 5 && a < 10 )) && echo "(( a > 5 && a < 10 ))"

5 -eq 05: true (numeric)
5 == 05: false (string)
7 -ge 5
(( a > 5 && a < 10 ))


In [17]:
%%bash expect_fail
[ 3 -eq 3.0 ]

script: line 1: [: 3.0: integer expression expected
[exit status 2]


In [18]:
%%bash
# an unset or empty variable is also an error in a numeric test: guard it
count=""
[ "${count:-0}" -eq 0 ] && echo "empty treated as 0 with a default"

empty treated as 0 with a default


## 09. `if` / `elif` / `else` in practice
- `elif` chains avoid deep nesting. The first branch whose condition succeeds runs and the rest are skipped.
- You can put several commands in a condition. The status of the last one counts.
- Put `then` on the same line with `;` or on its own line. Keep a consistent style.

In [19]:
%%bash
grade() {
  local n=$1
  if [ "$n" -ge 90 ]; then echo "$n -> A"
  elif [ "$n" -ge 80 ]; then echo "$n -> B"
  elif [ "$n" -ge 70 ]; then echo "$n -> C"
  else echo "$n -> below C"
  fi
}
grade 95; grade 85; grade 72; grade 10

95 -> A
85 -> B
72 -> C
10 -> below C


In [20]:
%%bash
# several commands in the condition: the last one decides
if echo "setup runs first"; true; then echo "last command succeeded"; fi
# a block as a condition
if { grep -q a <<< "abc" && grep -q z <<< "abc"; }; then echo never; else echo "both greps must match: they did not"; fi

setup runs first
last command succeeded
both greps must match: they did not


## 10. `&&`, `||` chains and short-circuit idioms
- `A && B` runs `B` only if `A` succeeded. `A || B` runs `B` only if `A` failed.
- Good for guards: `[ -f file ] || { echo missing >&2; exit 1; }`.
- Idiom: default value with `${var:-x}` is cleaner than `[ -z "$var" ] && var=x`.
- `[ cond ] && action` as the LAST command of a function or script returns status 1 when `cond` is false. That can make the whole script look like it failed, and breaks `set -e` scripts. Write `if` instead.

In [21]:
%%bash
[ -d / ] && echo "/ is a directory"
[ -d /nonexistent ] || echo "/nonexistent is not a directory"
cd /nonexistent 2>/dev/null || echo "cd failed, handled"
mkdir -p proj && cd proj && echo "inside proj: ${PWD##*/}" && cd ..

/ is a directory
/nonexistent is not a directory
cd failed, handled
inside proj: proj


In [22]:
%%bash
# the last-command trap
check() { [ -f nosuchfile ] && echo "found"; }
check
echo "status of check: $?   <- looks like a failure, but nothing went wrong"
check2() { if [ -f nosuchfile ]; then echo "found"; fi; }
check2
echo "status of check2: $?"

status of check: 1   <- looks like a failure, but nothing went wrong
status of check2: 0


## 11. `if ! cmd` and exit-status-driven flow
- `! cmd` flips the status: success becomes failure and the reverse.
- Prefer `if ! cmd; then ...` to running the command and checking `$?` on the next line. It is shorter and cannot be broken by something inserted in between.
- A function's status is the status of its last command, or whatever `return N` gives. Return codes make functions usable in `if`.
- `$?` can be tested directly, but only when you need the specific number (use `case $? in`).

In [23]:
%%bash
if ! grep -q "needle" <<< "haystack"; then echo "needle not found"; fi
is_even() { (( $1 % 2 == 0 )); }
for n in 3 4; do
  if is_even "$n"; then echo "$n is even"; else echo "$n is odd"; fi
done

needle not found
3 is odd
4 is even


In [24]:
%%bash
# distinguish different failures by number
grep -q x /nonexistent_file 2>/dev/null
case $? in
  0) echo "found" ;;
  1) echo "no match" ;;
  *) echo "error (status 2 or more): grep could not read the file" ;;
esac

error (status 2 or more): grep could not read the file


## 12. `case`: patterns, alternatives and ordering
- `case WORD in pattern) commands ;; esac` runs the first branch whose glob pattern matches. The rest are skipped.
- Patterns use glob syntax: `*`, `?`, `[a-z]`. `a|b)` means "a or b". `*)` is the catch-all and goes last.
- `;;` ends a branch (the last branch may omit it). Forgetting it between branches is a syntax error, not a fall-through.
- bash 4 adds `;&` (fall through to the next branch's commands) and `;;&` (keep testing the following patterns). Bash 3.2 does not have them.
- Quoting a pattern makes it literal. `case` does not split or glob `WORD`, so quotes there are mostly optional.

In [25]:
%%bash
kind() {
  case "$1" in
    [0-9]*)          echo "$1: starts with a digit" ;;
    *.txt | *.md)    echo "$1: text file (two alternatives in one branch)" ;;
    y | Y | yes)     echo "$1: affirmative" ;;
    [A-Z]*)          echo "$1: starts with an uppercase letter" ;;
    "")              echo "(empty string)" ;;
    *)               echo "$1: something else" ;;
  esac
}
kind 42abc; kind notes.md; kind Y; kind Zed; kind ""; kind '?x'

42abc: starts with a digit
notes.md: text file (two alternatives in one branch)
Y: affirmative
Zed: starts with an uppercase letter
(empty string)
?x: something else


In [26]:
%%bash
# order matters: the first match wins
case "banana" in
  b*)      echo "matched b* first" ;;
  banana)  echo "never reached" ;;
esac
# the status of case is the status of the last command run (0 if nothing matched)
case "zzz" in a) : ;; esac; echo "no branch ran, status $?"

matched b* first
no branch ran, status 0


Fall-through alternatives. Bash 3.2 does not have `;&` / `;;&`, so with older bash use `|` or an `if` chain.

```bash
# needs bash 4+ (not run here)
case $x in
  a) echo "a" ;&        # fall through into the next branch
  b) echo "b" ;;
  c*) echo "c" ;;&      # keep testing the following patterns
  *d) echo "ends with d" ;;
esac
```

In [27]:
%%bash
# bash 3.2 way to share work between branches: just use alternatives or a function
level() {
  case "$1" in
    debug)           echo "debug: very verbose" ;;
    info | notice)   echo "$1: normal" ;;
    warn | error)    echo "$1: attention needed" ;;
    *)               echo "$1: unknown level" >&2; return 1 ;;
  esac
}
level info; level warn; level nope; echo "status: $?"

info: normal
warn: attention needed
nope: unknown level
status: 1


## 13. `for` loops: lists, ranges, C-style
- `for x in a b c; do ... done` loops over words. The words can come from a literal list, a brace range, a glob or `"$@"`.
- `{1..5}` is expanded before variables, so `{1..$n}` does not work (see `01-Fundamentals`). Use `seq` or a C-style loop. `{1..5..2}` (step) needs bash 4.
- C-style: `for (( i=0; i<3; i++ )); do ... done` is bash only.
- After the loop, the variable keeps its last value.
- `for x; do ... done` (no `in` part) loops over `"$@"`.

In [28]:
%%bash
for c in red green blue; do echo "color: $c"; done
for i in {1..3}; do echo "brace range: $i"; done
for (( i = 0; i < 3; i++ )); do echo "C-style: i=$i"; done
echo "after the loop i=$i"

color: red
color: green
color: blue
brace range: 1
brace range: 2
brace range: 3
C-style: i=0
C-style: i=1
C-style: i=2
after the loop i=3


In [29]:
%%bash
n=4
for i in $(seq 1 "$n"); do printf '%s ' "$i"; done; echo "<- seq with a variable"
for (( i = n; i > 0; i -= 2 )); do printf '%s ' "$i"; done; echo "<- counting down by 2"

1 2 3 4 <- seq with a variable
4 2 <- counting down by 2


```bash
# needs bash 4+ (not run here)
for i in {1..10..3}; do echo "$i"; done    # 1 4 7 10
```

## 14. Looping over `"$@"` and over words safely
- `"$@"` keeps each argument as its own word, even with spaces. `$@` unquoted splits them again. `"$*"` joins all arguments into one string.
- A string with spaces is not a list. If you need a list, use an array (see `04-Strings-And-Arrays`) or a quoted list.
- Do not use a space-separated string to hold filenames.

In [30]:
%%bash
show() {
  echo "-- called with $# arguments"
  echo "good:"; for a in "$@"; do echo "  [$a]"; done
  echo "bad (unquoted):"; for a in $@; do echo "  [$a]"; done
}
show one "two words" three

-- called with 3 arguments
good:
  [one]
  [two words]
  [three]
bad (unquoted):
  [one]
  [two]
  [words]
  [three]


## 15. Looping over files: globs, no-match and odd names
- `for f in *.txt` is the right way to visit files. Each match is already one word, so spaces are safe as long as you quote `"$f"` inside.
- If nothing matches, the loop runs once with the literal text `*.txt`. Guard with `[ -e "$f" ] || continue`, or enable `nullglob`.
- `nullglob` (bash 3.2 OK) makes unmatched globs vanish. Remember to turn it off again so later globs behave normally (for example `ls *.zip` with no match would then list the current directory).
- A file named `-n` or `-rf` looks like an option. Prefix globs with `./` (`./*`) or use `--`.
- Never use `for f in $(ls)`: it splits names on whitespace and mangles odd characters.

In [31]:
%%bash
mkdir files && cd files
touch a.txt b.txt "c d.txt"
for f in *.txt; do echo "file: [$f]"; done
echo "--- no match, no guard:"
for f in *.zip; do echo "loop ran with: [$f]"; done
echo "--- guard with -e:"
for f in *.zip; do [ -e "$f" ] || continue; echo "never"; done; echo "guard: nothing printed"

file: [a.txt]
file: [b.txt]
file: [c d.txt]
--- no match, no guard:
loop ran with: [*.zip]
--- guard with -e:
guard: nothing printed


In [32]:
%%bash
echo "--- nullglob:"
shopt -s nullglob
for f in *.zip; do echo "never"; done; echo "nullglob: loop body did not run"
shopt -u nullglob
echo "--- for f in \$(ls) splits on spaces:"
for f in $(ls); do echo "[$f]"; done
echo "--- for f in * is correct:"
for f in *; do echo "[$f]"; done

--- nullglob:
nullglob: loop body did not run
--- for f in $(ls) splits on spaces:
[a.txt]
[b.txt]
[c]
[d.txt]
--- for f in * is correct:
[a.txt]
[b.txt]
[c d.txt]


In [33]:
%%bash
# a file whose name starts with a dash
touch ./-n
echo "plain *:   $(echo *)    <- echo ate the file -n as its own option"
echo "with ./*:  $(echo ./*)"
rm -f -- ./-n
cd ..

plain *:   a.txt b.txt c d.txt    <- echo ate the file -n as its own option
with ./*:  ./-n ./a.txt ./b.txt ./c d.txt


## 16. `while` and `until`
- `while cmd; do ...` repeats while `cmd` succeeds. `until cmd; do ...` repeats until `cmd` succeeds.
- They are two ways to write the same loop, pick whichever reads better.
- Always make sure something changes inside the loop, or it never ends. A safety counter is cheap insurance.
- `while true; do ... break; done` is the "loop until I say stop" pattern. `:` is a no-op that always succeeds, also usable instead of `true`.

In [34]:
%%bash
count=0
while [ "$count" -lt 3 ]; do echo "while: $count"; count=$((count + 1)); done
count=0
until [ "$count" -ge 3 ]; do echo "until: $count"; count=$((count + 1)); done

while: 0
while: 1
while: 2
until: 0
until: 1
until: 2


In [35]:
%%bash
# loop with a safety limit and an early break
tries=0
while true; do
  tries=$((tries + 1))
  if [ "$tries" -ge 5 ]; then echo "giving up after $tries tries"; break; fi
  [ "$tries" -eq 3 ] && { echo "succeeded on try $tries"; break; }
done

succeeded on try 3


## 17. `(( ))` arithmetic loops and conditions
- `(( expr ))` is a command, so it works directly as a loop or `if` condition: `while (( i < 5 ))`.
- Inside `(( ))` variables do not need `$`, and `<`, `>`, `==`, `&&` work with the meaning you expect for numbers.
- It evaluates to status 0 when the result is non-zero (true) and status 1 when the result is 0 (false).
- Watch `(( i++ ))` starting from 0: the value of the expression is 0, so the status is 1. Newer bash versions (documented behaviour, not testable on this 3.2) stop a `set -e` script there. Bash 3.2 quietly ignores it for `(( ))`, but the same trap bites with `let`. `(( ++i ))` or `i=$((i+1))` avoid it.

In [36]:
%%bash
i=0
while (( i < 3 )); do echo "i=$i"; (( i += 1 )); done
total=0
for (( n = 1; n <= 5; n++ )); do (( total += n )); done
echo "sum 1..5 = $total"
if (( total > 10 && total % 5 == 0 )); then echo "total is big and divisible by 5"; fi

i=0
i=1
i=2
sum 1..5 = 15
total is big and divisible by 5


In [37]:
%%bash fresh expect_fail
set -e
i=0
(( i++ ))          # value 0 -> status 1
echo "(( i++ )) did not stop this bash 3.2 script; the documented behaviour of newer bash is to stop here"
let "i = 0"        # same trap with let: status 1 and set -e does stop it
echo "never printed"

(( i++ )) did not stop this bash 3.2 script; the documented behaviour of newer bash is to stop here
[exit status 1]


## 18. `while read -r line`: reading lines the right way
- `while IFS= read -r line; do ...; done < file` reads a file line by line.
- `IFS=` stops `read` from trimming leading and trailing whitespace. `-r` stops it from treating backslashes as escapes. Use both by default.
- Redirect from a file (`< file`) or a here-string (`<<< "$var"`) into the loop so it runs in the current shell.
- A last line without a trailing newline makes `read` return failure, so the loop skips it. Add `|| [ -n "$line" ]`.
- Inside a `while read` loop, any command that reads stdin steals data from the file. Use `< /dev/null` on such commands, or another file descriptor.

In [38]:
%%bash
printf '  indented  line\nback\\slash\nplain\n' > lines.txt
echo "--- plain read (bad defaults):"
while read line; do echo "[$line]"; done < lines.txt
echo "--- IFS= read -r (good):"
while IFS= read -r line; do echo "[$line]"; done < lines.txt

--- plain read (bad defaults):
[indented  line]
[backslash]
[plain]
--- IFS= read -r (good):
[  indented  line]
[back\slash]
[plain]


In [39]:
%%bash
printf 'first\nsecond-no-newline' > nonl.txt
echo "--- missing newline, naive:"
while IFS= read -r line; do echo "[$line]"; done < nonl.txt
echo "--- missing newline, handled:"
while IFS= read -r line || [ -n "$line" ]; do echo "[$line]"; done < nonl.txt

--- missing newline, naive:
[first]
--- missing newline, handled:
[first]
[second-no-newline]


In [40]:
%%bash
# splitting fields while reading: set IFS just for read
printf 'ada:36:london\nalan:41:wilmslow\n' > people.txt
while IFS=: read -r name age city; do echo "$name is $age and lives in $city"; done < people.txt
# a command inside the loop that reads stdin eats the file
printf 'one\ntwo\nthree\n' > three.txt
echo "--- cat inside the loop steals the rest of the input:"
while IFS= read -r l; do echo "got [$l]"; cat >/dev/null; done < three.txt
echo "--- protected with </dev/null:"
while IFS= read -r l; do echo "got [$l]"; cat </dev/null; done < three.txt
rm -f lines.txt nonl.txt people.txt three.txt

ada is 36 and lives in london
alan is 41 and lives in wilmslow
--- cat inside the loop steals the rest of the input:
got [one]
--- protected with </dev/null:
got [one]
got [two]
got [three]


## 19. The pipeline subshell trap
- Each part of a pipeline runs in its own subshell (a copy of the shell). So `cmd | while read ...` changes variables only inside the copy.
- After the loop the counter is back to what it was. No error is shown. The script is just silently wrong.
- Fixes that work in bash 3.2: redirect from a file or here-string, or use process substitution `< <(cmd)`.
- bash 4.2+ has `shopt -s lastpipe` (run the last pipeline stage in the current shell, only when job control is off). Not available here.
- Other fix: do all the work inside the loop (print the total at the end of the same subshell).

In [41]:
%%bash
n=0
printf 'a\nb\nc\n' | while read -r l; do n=$((n + 1)); done
echo "BAD  (pipe):                 n=$n"
n=0
while read -r l; do n=$((n + 1)); done <<< "$(printf 'a\nb\nc\n')"
echo "GOOD (here-string):          n=$n"
n=0
while read -r l; do n=$((n + 1)); done < <(printf 'a\nb\nc\n')
echo "GOOD (process substitution): n=$n"

BAD  (pipe):                 n=0
GOOD (here-string):          n=3
GOOD (process substitution): n=3


In [42]:
%%bash
# do everything inside the subshell when you only need a summary
printf 'a\nb\nc\n' | { n=0; while read -r l; do n=$((n + 1)); done; echo "GOOD (same subshell): n=$n"; }

GOOD (same subshell): n=3


```bash
# needs bash 4.2+ (not run here)
shopt -s lastpipe
printf 'a\nb\n' | while read -r l; do n=$((n+1)); done
echo "$n"
```

## 20. Looping over command output safely
- `for x in $(cmd)` splits on whitespace and expands globs in the output. Only use it for output you fully control (such as `seq`).
- For lines: `while IFS= read -r line; do ...; done < <(cmd)`.
- For filenames from `find`: use NUL separators. `find ... -print0 | while IFS= read -r -d '' f` works in bash 3.2 and with BSD `find`.
- Often `find ... -exec cmd {} +` or `xargs` replaces the loop completely (see `02-Streams-And-Text/02-Text-Processing-Tools`, `03-Files-And-System/01-File-System-Ops`).

In [43]:
%%bash
mkdir tree && touch "tree/one.log" "tree/two words.log" "tree/three.txt"
echo "--- BAD: for over \$(find):"
for f in $(find tree -name '*.log' | sort); do echo "[$f]"; done
echo "--- GOOD: NUL-separated:"
find tree -name '*.log' -print0 | sort -z | while IFS= read -r -d '' f; do echo "[$f]"; done
echo "--- GOOD: find -exec (no loop):"
find tree -name '*.log' -exec basename {} \; | sort

--- BAD: for over $(find):
[tree/one.log]
[tree/two]
[words.log]
--- GOOD: NUL-separated:
[tree/one.log]
[tree/two words.log]
--- GOOD: find -exec (no loop):
one.log
two words.log


In [44]:
%%bash
# lines of output, safely, keeping variables in the current shell
count=0
while IFS= read -r l; do count=$((count + 1)); done < <(printf 'x y\nz\n')
echo "lines: $count"
rm -rf tree

lines: 2


## 21. `break`, `continue` and `break N`
- `break` leaves the innermost loop. `continue` jumps to its next iteration.
- `break N` / `continue N` act on the Nth enclosing loop (1 is the innermost).
- They also work in `while` and `until`. Calling them outside a loop is an error (or ignored, depending on the shell).

In [45]:
%%bash
for n in 1 2 3 4 5; do
  [ "$n" -eq 2 ] && continue
  [ "$n" -eq 4 ] && break
  echo "n=$n"
done

n=1
n=3


In [46]:
%%bash
# searching a grid: leave BOTH loops when found
for row in 1 2 3; do
  for col in 1 2 3; do
    if [ "$row$col" = "22" ]; then echo "found 22, leaving both loops"; break 2; fi
    echo "checked $row$col"
  done
done
echo "--- continue 2 skips to the next row:"
for row in 1 2; do
  for col in 1 2 3; do
    [ "$col" -eq 2 ] && continue 2
    echo "row $row col $col"
  done
done

checked 11
checked 12
checked 13
checked 21
found 22, leaving both loops
--- continue 2 skips to the next row:
row 1 col 1
row 2 col 1


## 22. `select`: simple numbered menus
- `select var in list; do ... done` prints a numbered menu to stderr, shows the `PS3` prompt and reads a choice from stdin.
- `$REPLY` holds what the user typed. `var` is the chosen item, or empty for an invalid number.
- The loop repeats until `break` or end-of-input. There is no `Quit` item: you add it yourself.
- In scripts and notebooks stdin is not a person, so feed it with a here-string. Redirect the whole `select` loop.

In [47]:
%%bash
PS3="choice> "
select fruit in apple banana quit; do
  case "$REPLY" in
    1|2) echo "you picked $fruit" ;;
    3)   echo "bye"; break ;;
    *)   echo "invalid: [$REPLY]" ;;
  esac
done <<< $'2\n9\n1\n3'
echo "after select, status $?"

1) apple
2) banana
3) quit
choice> you picked banana
choice> invalid: [9]
choice> you picked apple
choice> bye
after select, status 0


In [48]:
%%bash
# at end of input select just stops
select x in a b; do echo "picked $x"; done <<< "1"
echo "status at EOF: $?"

1) a
2) b
choice> picked a
choice> 
status at EOF: 1


## 23. A menu-driven example, fed through a here-string
- A hand-written menu uses `while`, `read` and `case`. It gives full control over the prompt and the loop.
- Feeding input with `<<<` makes it testable and deterministic. Replace it with `< /dev/stdin` (or nothing) for interactive use.
- Separate "read a choice" from "act on it", and always handle empty and unknown input.

In [49]:
%%bash
balance=100
menu() {
  while IFS= read -r choice; do
    case "$choice" in
      b|balance)   echo "balance: $balance" ;;
      d*)          amount=${choice#d }
                   if [[ $amount =~ ^[0-9]+$ ]]; then balance=$((balance + amount)); echo "deposited $amount"
                   else echo "usage: d <number>"; fi ;;
      "")          continue ;;
      q|quit)      echo "bye"; break ;;
      *)           echo "unknown command: $choice" ;;
    esac
  done
}
menu <<< $'b\nd 50\nb\nd abc\n\nhelp\nq\nb'
echo "final balance (loop ran in current shell): $balance"

balance: 100
deposited 50
balance: 150
usage: d <number>
unknown command: help
bye
final balance (loop ran in current shell): 150


## 24. Retry and polling loops driven by exit status
- Loops are often "do something until it works": the command's status is the condition.
- `until cmd; do ...; done` and `while ! cmd; do ...; done` are equivalent.
- Always cap the number of attempts and sleep between them. Here `sleep` is skipped to keep the demo instant.

In [50]:
%%bash
attempt=0
flaky() { attempt=$((attempt + 1)); [ "$attempt" -ge 3 ]; }
max=5
until flaky; do
  echo "attempt $attempt failed"
  [ "$attempt" -ge "$max" ] && { echo "giving up"; break; }
  # sleep 1
done
echo "finished after $attempt attempt(s)"

attempt 1 failed
attempt 2 failed
finished after 3 attempt(s)


In [51]:
%%bash
attempt=0
while ! flaky; do echo "while !: attempt $attempt failed"; done
echo "succeeded on attempt $attempt"

while !: attempt 1 failed
while !: attempt 2 failed
succeeded on attempt 3


## 25. Bad vs good: a quick checklist
- Quote variables in `[ ]`. Prefer `[[ ]]` in bash scripts.
- Use `=` (not `-eq`) for strings and `-eq` (not `=`) for integers.
- Loop over globs and `"$@"`, never over `$(ls)` or an unquoted list of names.
- Read lines with `IFS= read -r`, fed by `<` or `< <(...)`, not by a pipe, when you need the variables afterwards.
- Guard the no-match glob. Use `./*` for arbitrary filenames.
- Prefer `if`/`else` over `A && B || C`.

In [52]:
%%bash
printf 'file one.txt\n' > list.txt
echo "BAD:"
for f in $(cat list.txt); do echo "  [$f]"; done
echo "GOOD:"
while IFS= read -r f; do echo "  [$f]"; done < list.txt
rm -f list.txt

BAD:
  [file]
  [one.txt]
GOOD:
  [file one.txt]
